# Steerling-8B: Text Generation

Steerling is a **causal diffusion language model**. Unlike standard autoregressive models that
generate one token at a time left-to-right, Steerling generates text in **blocks of 64 tokens** (matching the block-causal attention window).

Within each block, tokens start fully masked and are iteratively **unmasked** over multiple denoising
steps. At each step, the model scores all masked positions and unmasks the highest-confidence ones first.
This confidence-based ordering means generation is **not strictly left-to-right** —
the model can fill in the middle of a sentence before the end.

**Requirements:** GPU with >= 18 GB VRAM (A100, A6000, RTX 4090)

## 1. Load Model

We load the model via HuggingFace `AutoModel` and wrap it with `SteerlingGenerator`,
which handles the block-by-block denoising loop.

First run downloads ~17 GB from HuggingFace Hub; subsequent runs load from cache.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from steerling import SteerlingGenerator, GenerationConfig

model_id = "guidelabs/steerling-8b"

model = AutoModel.from_pretrained(model_id, trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

generator = SteerlingGenerator.from_model(model, tokenizer, device="cuda")
print(generator)
print(f"\nBlock size: {generator.diff_block_size} tokens")
print(f"Interpretable: {generator.is_interpretable}")

## 2. Generate Text

`generator.generate()` takes a prompt string and a `GenerationConfig`, and returns the generated text.

Key parameters:
- **`max_new_tokens`** — how many tokens to generate. Must be divisible by `block_length` (default 64).
- **`steps`** — total denoising steps, split evenly across blocks. More steps = more refinement per block.
- **`temperature`** — controls randomness. `0.0` is greedy (deterministic); higher values add Gumbel noise for diversity.

When `steps == max_new_tokens`, each denoising step unmasks exactly one token per block.

In [ ]:
prompt = "The key to understanding artificial intelligence is"
config = GenerationConfig(max_new_tokens=128, steps=128, temperature=0.4, seed=42)

text = generator.generate(prompt, config)

print(f"Prompt:    {prompt}")
print(f"Generated: {text}")

### Effect of `steps` on speed and quality

The `steps` parameter controls how many denoising iterations are used. When `steps == max_new_tokens`,
each step unmasks exactly one token per block (highest quality). Reducing `steps` unmasks multiple
tokens per step, faster generation but lower quality, since the model makes more decisions per step
with less context. If `steps` is not set, it defaults to `max_new_tokens`.

In [ ]:
prompt = "The key to understanding artificial intelligence is"



# Max steps = best quality (1 token unmasked per step)
config_best = GenerationConfig(max_new_tokens=128, steps=128, temperature=0.4, seed=42)
text_best = generator.generate(prompt, config_best)

print("=== 16 steps (fast) ===")
print(text_fast)
print("\n=== 128 steps (best quality) ===")
print(text_best)

## 3. Special Tokens and Early Stopping

Steerling uses a custom tokenizer (tiktoken cl100k_base) with 4 additional special tokens:

| Token | ID | Purpose |
|-------|------|---------|
| `<\|endofchunk\|>` | 100279 | End-of-chunk — separates conceptual segments in training data |
| `<\|mask\|>` | 100280 | Mask token used during diffusion denoising |

The **`<|endofchunk|>`** token: the model was trained to insert it
between conceptual segments of text. You can use it as a **stop token** to tell the generator
to halt after one chunk instead of filling all `gen_length` tokens.

In [ ]:
prompt = "Once upon a time"

# Without stop tokens — generates all 256 tokens
text_full = generator.generate(
    prompt,
    GenerationConfig(max_new_tokens=256, steps=256, temperature=0.4, seed=42),
)

# With stop tokens — stops at first <|endofchunk|>
text_stopped = generator.generate(
    prompt,
    GenerationConfig(
        max_new_tokens=256,
        steps=256,
        temperature=0.4,
        seed=42,
        stop_tokens=[tokenizer.endofchunk_token_id],
    ),
)

print("=== Full generation ===")
print(text_full)
print("\n=== With endofchunk stop ===")
print(text_stopped)

## Generation Parameters Reference

| Parameter          | Default              | Description                                              |
|--------------------|----------------------|----------------------------------------------------------|
| `max_new_tokens`   | 1024                 | Number of tokens to generate                             |
| `steps`            | `max_new_tokens`     | Total denoising steps (split evenly across blocks)       |
| `temperature`      | 1.2                  | Sampling temperature (0.0 = greedy)                      |
| `cfg_scale`        | 0.0                  | Classifier-free guidance scale (0 = disabled)            |
| `seed`             | None                 | Random seed for reproducibility                          |
| `stop_tokens`      | None                 | Token IDs that trigger early stop between blocks         |

**Constraints:**
- `max_new_tokens` must be divisible by `block_length` (default 64)
- `steps` must be divisible by the number of blocks (`max_new_tokens / block_length`)